# pcap2mtam

- Read and process a (large) number of PCAP files, creating an MTAM matrix per file.
- Flatten and save the MTAMs as well as the maching backend

In [ ]:
from pathlib import Path
import pcap_tools as pt
import re
import numpy as np
from tqdm import tqdm

AGENT_TYPE = "research_agent"  # weather_agent, no-tools_agent
SET_TYPE = "all"  # train, test or all (all will be split twice in RandomForestClassifier)
MAX_MATRIX_LEN = 4800 # 4800 for research_agent, 1800 for weather_agent and no-tools_agent
PCAP_DIR = f"/Users/tarik/data/tmp/research_random_120/pcap/1/"
CLIENT_IP = "172.17.0.2"
WINDOW_SIZE = 0.05  # in seconds

In [ ]:
files = [p for p in Path(PCAP_DIR).iterdir() 
         if p.is_file() and p.suffix.lower() in (".pcap", ".pcapng")]

mtams = []
targets = []

for p in tqdm(files, desc="Processing PCAPs"):
    backend, model, dt = pt.extract_agent_from_filename(p.name)
    trace = pt.pcap_to_trace_scapy(str(p), CLIENT_IP)

    mtam = pt.build_mtam(trace, window_size=WINDOW_SIZE, num_windows=MAX_MATRIX_LEN)
    mtams.append(mtam.reshape(-1))
    targets.append(pt.get_code(backend))

In [ ]:
# Create and check numpy arrays

X = np.vstack(mtams)
y = np.array(targets)
assert X.shape[0] == y.shape[0]
assert np.all(y >= 0)

np.savez_compressed(f"../ml/datasets/{AGENT_TYPE}_{SET_TYPE}.npz", dataset=X, label=y)